In [38]:
import os
import numpy as np
import re
import warnings
warnings.filterwarnings("ignore")
from tqdm import tqdm 

# Decorter packages
from typing import List

# dataframe packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Tensorflow packages
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, 
                                    Input, 
                                    LSTM, 
                                    Dropout, 
                                    Conv1D, 
                                    MaxPooling1D,
                                    Bidirectional,  
                                    Flatten
                                )
from tensorflow.keras.optimizers import (Adam, 
                                         AdamW)
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report

In [2]:
print("TensorFlow version:", tf.__version__)
print("Available physical devices:")
print(tf.config.list_physical_devices())

print("\nIs MPS available?")
print(tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.19.0
Available physical devices:
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Is MPS available?
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


#### Step -1 :  Load the Dataset

In [3]:
data = np.load("../eeg_processed/eeg_data.npz")
data["X"].shape, data["y"].shape


X = data["X"]
# Transpose to match (num_trails, n_samples, n_channels)
X = np.transpose(X, (0, 2, 1))

y = data["y"]

print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")

Shape of X: (1799, 500, 16)
Shape of y: (1799,)


#### Step -2 : Normalize Dataset

In [4]:
# Global Normalization
X = (X - np.mean(X)) / np.std(X)

print(f"Shape of X: {X.shape}")

Shape of X: (1799, 500, 16)


#### Step -3 : One Hot Encode labels (for classification)

In [5]:
y[y == 2] = 0 
y[y == 3] = 1
y[y == 4] = 2 
y[y == 5] = 3
y[y == 6] = 4 
y[y == 7] = 5

print(np.unique(y))

[0 1 2 3 4 5]


In [6]:
np.unique_counts(y) # It's a balanced dataset

UniqueCountsResult(values=array([0, 1, 2, 3, 4, 5]), counts=array([299, 300, 300, 300, 300, 300]))

In [7]:
y = to_categorical(y, num_classes=len(np.unique(y)))

#### Step 4  Creating Tensorflow Dataset

In [8]:
data = tf.data.Dataset.from_tensor_slices((X, y))

2025-10-08 20:41:12.727025: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Pro
2025-10-08 20:41:12.727063: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 18.00 GB
2025-10-08 20:41:12.727069: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 6.66 GB
I0000 00:00:1759974072.727084 3519226 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1759974072.727110 3519226 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [9]:
for x_batch, y_batch in data.take(1):
    print(x_batch.shape)
    print(y_batch.shape)

(500, 16)
(6,)


2025-10-08 20:41:13.395503: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


#### Step 5: Splitting the Tensorflow Dataset into Train, Test and Valid

In [10]:
train_split_ratio = 0.7
val_split_ratio = 0.15
test_split_ratio = 0.15

# Size of each split
train_size = int(train_split_ratio * X.shape[0])
val_size = int(val_split_ratio * X.shape[0])
test_size = int(test_split_ratio * X.shape[0])

# Shuffle the dataset first for a random split
data = data.shuffle(buffer_size= X.shape[0])

training_dataset = data.take(train_size)
val_dataset = data.take(val_size)
test_dataset = data.take(test_size)

print(f"Train dataset size: {len(list(training_dataset.as_numpy_iterator()))}")
print(f"Val dataset size: {len(list(val_dataset.as_numpy_iterator()))}")
print(f"Test dataset size: {len(list(test_dataset.as_numpy_iterator()))}")

Train dataset size: 1259
Val dataset size: 269
Test dataset size: 269


2025-10-08 20:41:14.988156: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2025-10-08 20:41:15.028768: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [11]:
training_dataset = training_dataset.batch(32) # Training
val_dataset = val_dataset.batch(32) # Validation
test_dataset = test_dataset.batch(32) # Testing

#### Step 6: LSTM Model Architecture

In [39]:
class LSTMModel(Model):

    def __init__(self, num_classes):
        super(LSTMModel, self).__init__()

        # Bidirectional
        self.multi_dir = Bidirectional(LSTM(
            units = 256, 
            input_shape = (500, 16),
            return_sequences = True
        ))

        # LSTM and DropOut layer with 80 hidden units
        self.lstm = LSTM(units=256, 
                         return_sequences=True)
        self.dropout = Dropout(rate=0.2)

        # LSTM layer 2
        self.lstm_layer_2 = LSTM(units=256, 
                         return_sequences=True)
        
        # LSTM layer 3
        self.lstm_layer_3 = LSTM(units=256, 
                         return_sequences=False)

        # Fully connected layer (2 units for binary classification)
        self.fc_1 = Dense(units=256, activation='leaky_relu')
        self.fc_2 = Dense(units=256, activation='leaky_relu')
        self.fc_2 = Dense(units=256, activation='leaky_relu')
        self.fc_2 = Dense(units=128, activation='leaky_relu')
        self.out = Dense(units=num_classes, activation='softmax')  # softmax for multi-class

    def call(self, inputs, training=False):
        #multi-directional lstm
        x = self.multi_dir(inputs)

        # lstm layer 1
        x = self.lstm(x)
        x = self.dropout(x, training=training)

        # lstm layer 2
        x = self.lstm_layer_2(x)
        x = self.dropout(x, training=training)

        # lstm layer 3
        x = self.lstm_layer_3(x)
        x = self.dropout(x, training=training)
        
        # Dense layer 1
        x = self.fc_1(x)
        x = self.fc_2(x)

        # Output layer
        out = self.out(x)
        return out

# Instantiate model
num_classes = 6  # Change if you have more classes
lstm_model = LSTMModel(num_classes=num_classes)

In [40]:
print(lstm_model.summary())

f1_metric = tf.keras.metrics.F1Score(average='macro')

# Compile the Model
lstm_model.compile(optimizer= Adam(), 
                        loss = "categorical_crossentropy", 
                        metrics = ["accuracy", f1_metric])

Model: "lstm_model_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_16 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_17 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_18 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [41]:
lstm_model.summary()

Model: "lstm_model_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_16 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_17 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_18 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [42]:
history = lstm_model.fit(
    training_dataset,
    epochs = 40, 
    batch_size = 32, 
    validation_data = val_dataset
)

Epoch 1/40
40/40 ━━━━━━━━━━━━━━━━━━━━ 13s 256ms/step - accuracy: 0.1565 - f1_score: 0.1310 - loss: 1.7978 - val_accuracy: 0.1636 - val_f1_score: 0.0939 - val_loss: 1.7923
Epoch 2/40
40/40 ━━━━━━━━━━━━━━━━━━━━ 10s 239ms/step - accuracy: 0.1535 - f1_score: 0.1124 - loss: 1.7953 - val_accuracy: 0.1599 - val_f1_score: 0.0764 - val_loss: 1.8016
Epoch 3/40
40/40 ━━━━━━━━━━━━━━━━━━━━ 10s 239ms/step - accuracy: 0.1880 - f1_score: 0.1256 - loss: 1.7837 - val_accuracy: 0.2045 - val_f1_score: 0.1581 - val_loss: 1.7823
Epoch 4/40
40/40 ━━━━━━━━━━━━━━━━━━━━ 10s 240ms/step - accuracy: 0.1855 - f1_score: 0.1584 - loss: 1.7845 - val_accuracy: 0.1896 - val_f1_score: 0.1774 - val_loss: 1.7857
Epoch 5/40
40/40 ━━━━━━━━━━━━━━━━━━━━ 10s 239ms/step - accuracy: 0.1765 - f1_score: 0.1432 - loss: 1.7899 - val_accuracy: 0.2565 - val_f1_score: 0.2452 - val_loss: 1.7621
Epoch 6/40
40/40 ━━━━━━━━━━━━━━━━━━━━ 10s 239ms/step - accuracy: 0.1877 - f1_score: 0.1660 - loss: 1.7790 - val_accuracy: 0.1784 - val_f1_score: 

In [36]:
# Test Predictions
y_val_pred = lstm_model.predict(val_dataset)
y_val_pred = np.argmax(y_val_pred, axis = 1)

# Original Predictions
y_val_original = np.concatenate([y.numpy() for x, y in val_dataset])
y_val_original = np.argmax(y_val_original, axis = 1)

9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step


In [37]:
print(classification_report(y_val_original, y_val_pred))

              precision    recall  f1-score   support

           0       0.20      0.17      0.19        40
           1       0.15      0.25      0.19        40
           2       0.26      0.27      0.27        41
           3       0.12      0.18      0.14        34
           4       0.10      0.06      0.08        49
           5       0.34      0.25      0.29        65

    accuracy                           0.20       269
   macro avg       0.20      0.20      0.19       269
weighted avg       0.21      0.20      0.20       269

